# Deduplication — Removing Duplicate Rows

Remove duplicate records across the intermediate DataFrame with full control over column subsets.

`pytae` provides both the method-chaining accessor `df.pt.dedupe(*cols, keep='first')` and the functional verb `pt.dedupe(df, *cols)`. In the CLI, the equivalent flag is `-dedupe` (or `-dedupe COLS`).

| Operation | Pytae Method Chain | CLI Equivalent |
|---|---|---|
| All Columns | `df.pt.dedupe()` | `-dedupe` |
| Specific Columns | `df.pt.dedupe('col1', 'col2')` | `-dedupe "col1,col2"` |
| Keep Last / None | `df.pt.dedupe(keep='last')` | N/A |

---

In [ ]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt
import pandas as pd

penguins = pt.sample_data['penguins']
tips = pt.sample_data['tips']

## 1. Deduplicating Across All Columns (`.pt.dedupe()`)

Eliminate identical rows across all selected columns. The index is automatically reset:

In [ ]:
# Keep only unique species and island pairs
(
    penguins
    .pt.select("species", "island")
    .pt.dedupe()
)

## 2. Deduplicating by Specific Identifier Columns

Pass column names to restrict duplicate checks to specific identifiers, keeping the first matching record for all other columns:

In [ ]:
# Keep the first recorded penguin for each unique species and island combination
(
    penguins
    .pt.select("species", "island", "body_mass_g", "sex")
    .pt.dedupe("species", "island")
)

## 3. Chaining in Pipelines

`df.pt.dedupe()` flows seamlessly inside multi-step pipelines without dropping out of `.pt`:

In [ ]:
# Filter heavy penguins -> select columns -> deduplicate -> arrange
(
    penguins
    .pt.qry("body_mass_g > 4000")
    .pt.select("species", "island")
    .pt.dedupe()
    .pt.arrange("species", "island")
)

## 4. Keeping the Last Record (`keep='last'`)

Optionally retain the last occurrence instead of the first:

In [ ]:
# Retain the last record for each day and time
(
    tips
    .pt.select("day", "time", "total_bill", "tip")
    .pt.dedupe("day", "time", keep="last")
)

## 5. Functional Style (`pt.dedupe`)

Available as a standalone function:

In [ ]:
# Functional deduplication
pt.dedupe(penguins[["species", "island"]])